# Model B V1 Training Notebook

This notebook trains the shared multi-head attribute model for StyleSync.

Model purpose:

- take one clothing image
- predict recommendation-friendly attributes
- support `tops`, `bottomwear`, and `outerwear`
- use masked supervision so each category only trains relevant heads

This notebook is MPS-first for Apple Silicon and falls back to CPU automatically.


In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms


In [ ]:
NOTEBOOK_DIR = Path.cwd()
ATTRIBUTE_MODELING_ROOT = NOTEBOOK_DIR.parent
DEEPFASHION_ROOT = ATTRIBUTE_MODELING_ROOT.parent
DATA_ROOT = DEEPFASHION_ROOT / "data"
CSV_PATH = DATA_ROOT / "anno_fine_v2_common_items_material_merged.csv"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"DeepFashion root: {DEEPFASHION_ROOT}")
print(f"CSV exists: {CSV_PATH.exists()}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
SEED = 42
IMAGE_SIZE = 224
RESIZE_SIZE = 256
BATCH_SIZE = 32
NUM_WORKERS = 0
VAL_SIZE = 0.2
USE_OFFICIAL_TEST_AS_HOLDOUT = True
USE_WEIGHTED_MATERIAL_SAMPLER = True

# Two-stage training keeps V2 practical while still letting the backbone adapt.
STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 5
STAGE1_LR = 1e-3
STAGE2_HEAD_LR = 5e-4
STAGE2_BACKBONE_LR = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
LABEL_SMOOTHING = 0.0
EARLY_STOPPING_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 1
MAX_CLASS_WEIGHT = 4.0
IGNORE_INDEX = -100
SHARED_DIM = 512

CHECKPOINT_PATH = OUTPUT_DIR / "model_b_v2_material_merged_resnet50_best.pt"
VOCAB_PATH = OUTPUT_DIR / "model_b_v2_material_merged_label_vocabs.json"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"Stage 1 epochs: {STAGE1_EPOCHS} | Stage 2 epochs: {STAGE2_EPOCHS}")
print(f"Weighted material sampler enabled: {USE_WEIGHTED_MATERIAL_SAMPLER}")


## Head Definitions

The label sets below come directly from the V1 head spec.


In [ ]:
HEAD_SPECS = {
    # Each head is its own attribute task with a separate classifier.
    "pattern_family": ["solid", "graphic", "floral", "striped", "embroidered", "other"],
    "material_family": ["denim", "knit", "chiffon", "leather", "other"],
    "sleeve_family": ["sleeveless", "short_sleeve", "long_sleeve"],
}

CATEGORY_HEADS = {
    # This mapping is what turns the model into masked multi-task learning.
    # A head only contributes loss when it is relevant for the current garment category.
    "top": ["pattern_family", "material_family", "sleeve_family"],
    "bottom": ["pattern_family", "material_family"],
    "outerwear": ["pattern_family", "material_family", "sleeve_family"],
}

label_to_idx = {
    head: {label: idx for idx, label in enumerate(labels)}
    for head, labels in HEAD_SPECS.items()
}
idx_to_label = {
    head: {idx: label for idx, label in enumerate(labels)}
    for head, labels in HEAD_SPECS.items()
}

with VOCAB_PATH.open("w") as f:
    json.dump({"head_specs": HEAD_SPECS, "category_heads": CATEGORY_HEADS}, f, indent=2)

HEAD_SPECS


## V2 Material Merge Rationale

This notebook keeps the 3-head recommendation-focused setup, but upgrades `material_family` into a cleaner V2 schema by merging `faux` into `leather`.

Why `faux` was merged:
- `faux` was too rare as a standalone class.
- `faux` and `leather` overlap visually in this dataset.
- The model was paying a high complexity cost for a small class with weak recall.
- Merging them creates a stronger elevated-material signal, especially for outerwear.

Why these three heads remain:
- `pattern_family` is the strongest outfit-matching signal.
- `material_family` is noisy, but now more learnable after the merge.
- `sleeve_family` is useful for coverage, layering, and seasonality.

The V2 goal is to improve material prediction quality without adding more model complexity.


In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df["image_exists"] == True].copy()
df = df[df["category_group"].isin(CATEGORY_HEADS.keys())].copy()

for head, labels in HEAD_SPECS.items():
    df[head] = df[head].astype(str)
    if head == "material_family":
        # Defensive fallback in case an older CSV still contains faux labels.
        df[head] = df[head].replace({"faux": "leather"})
    applicable_mask = df["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
    # Non-applicable heads are marked explicitly so the dataset can ignore them later.
    df.loc[~applicable_mask, head] = "not_applicable"
    known_mask = df[head].isin(labels) | (df[head] == "not_applicable")
    unknown_mask = applicable_mask & (~known_mask)
    if unknown_mask.any():
        # Unknown applicable labels are collapsed into a safe fallback bucket.
        print(f"Collapsing {unknown_mask.sum()} rows of unknown {head} labels into 'other'")
        if "other" not in labels:
            raise ValueError(f"Head {head} has unknown labels but no 'other' fallback")
        df.loc[unknown_mask, head] = "other"

print(df.shape)
print(df["category_group"].value_counts().to_string())
display(df[["image_name", "category_group"] + list(HEAD_SPECS.keys())].head())


In [ ]:
for head, labels in HEAD_SPECS.items():
    applicable_mask = df["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
    df.loc[~applicable_mask, head] = "not_applicable"
    known_mask = df[head].isin(labels) | (df[head] == "not_applicable")
    unknown_mask = applicable_mask & (~known_mask)
    if unknown_mask.any():
        print(f"Collapsing {unknown_mask.sum()} remaining unknown {head} labels into 'other'")
        df.loc[unknown_mask, head] = "other"
    unknown = sorted(set(df.loc[applicable_mask, head].unique()) - set(labels))
    print(head, "unknown labels:", unknown)
    print(df[head].value_counts().to_string())
    print()
    assert len(unknown) == 0, f"Unexpected labels remain in {head}: {unknown}"


## Split Strategy

Use the official `test` rows as a holdout set and stratify a train/validation split on the remaining rows by `category_group`.


In [ ]:
if USE_OFFICIAL_TEST_AS_HOLDOUT and "split" in df.columns:
    train_val_df = df[df["split"] != "test"].copy()
    test_df = df[df["split"] == "test"].copy()
else:
    train_val_df = df.copy()
    test_df = df.iloc[0:0].copy()

train_df, val_df = train_test_split(
    train_val_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_val_df["category_group"],
)

print(f"Train rows: {len(train_df)}")
print(f"Val rows: {len(val_df)}")
print(f"Test rows: {len(test_df)}")
print(train_df["category_group"].value_counts().to_string())

def compute_majority_baselines(frame):
    baselines = {}
    for head in HEAD_SPECS:
        applicable_mask = frame["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
        values = frame.loc[applicable_mask, head].astype(str)
        if len(values) == 0:
            baselines[head] = float("nan")
            continue
        baselines[head] = values.value_counts(normalize=True).iloc[0]
    return baselines

train_majority_baselines = compute_majority_baselines(train_df)
val_majority_baselines = compute_majority_baselines(val_df)
test_majority_baselines = compute_majority_baselines(test_df)

pd.DataFrame({
    "train_majority_baseline": train_majority_baselines,
    "val_majority_baseline": val_majority_baselines,
    "test_majority_baseline": test_majority_baselines,
})


In [ ]:
def build_train_transform(image_size=224, resize_size=256):
    # Train-time augmentation is intentionally moderate so garment details stay recognizable.
    return transforms.Compose([
        transforms.Resize((resize_size, resize_size)),
        transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
        transforms.RandomRotation(5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def build_eval_transform(image_size=224, resize_size=256):
    return transforms.Compose([
        transforms.Resize((resize_size, resize_size)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

train_transform = build_train_transform(IMAGE_SIZE, RESIZE_SIZE)
eval_transform = build_eval_transform(IMAGE_SIZE, RESIZE_SIZE)

class ModelBAttributeDataset(Dataset):
    def __init__(self, frame, transform):
        self.df = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)

        category_group = row["category_group"]
        sample = {
            "image": image,
            "category_group": category_group,
            "image_name": row["image_name"],
        }

        for head, labels in HEAD_SPECS.items():
            applicable = head in CATEGORY_HEADS[category_group]
            # The mask records whether this sample should supervise the current head.
            sample[f"{head}_mask"] = torch.tensor(applicable, dtype=torch.bool)
            if applicable:
                value = row[head]
                if head == "material_family" and value == "faux":
                    value = "leather"
                if value not in label_to_idx[head]:
                    if "other" in label_to_idx[head]:
                        value = "other"
                    else:
                        raise KeyError(f"Unknown label for {head}: {value}")
                sample[f"{head}_target"] = torch.tensor(label_to_idx[head][value], dtype=torch.long)
            else:
                # IGNORE_INDEX turns off loss and metrics for irrelevant heads.
                sample[f"{head}_target"] = torch.tensor(IGNORE_INDEX, dtype=torch.long)

        return sample


def build_material_sampler(frame):
    material_counts = frame["material_family"].value_counts()
    sample_weights = frame["material_family"].map(lambda value: 1.0 / material_counts[value]).astype(float)
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights.to_numpy(), dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


train_dataset = ModelBAttributeDataset(train_df, transform=train_transform)
val_dataset = ModelBAttributeDataset(val_df, transform=eval_transform)
test_dataset = ModelBAttributeDataset(test_df, transform=eval_transform)

train_sampler = build_material_sampler(train_df) if USE_WEIGHTED_MATERIAL_SAMPLER else None
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
print(batch["image"].shape)
for head in HEAD_SPECS:
    print(head, batch[f"{head}_target"][:5], batch[f"{head}_mask"][:5])


## Model Definition


In [ ]:
class ModelBResNet50(nn.Module):
    def __init__(self, head_specs, shared_dim=512, dropout=0.3, freeze_backbone=True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        backbone = models.resnet50(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        # The projection is the shared representation that every head reads from.
        self.projection = nn.Sequential(
            nn.Linear(in_features, shared_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        # ModuleDict is the key multi-head change from a standard single-label ResNet.
        self.heads = nn.ModuleDict({
            head: nn.Linear(shared_dim, len(labels))
            for head, labels in head_specs.items()
        })

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        # A normal classifier would return one logits tensor.
        # This model returns one logits tensor per head.
        features = self.backbone(x)
        shared = self.projection(features)
        return {head: layer(shared) for head, layer in self.heads.items()}


def freeze_backbone(model):
    for param in model.backbone.parameters():
        param.requires_grad = False


def unfreeze_backbone_layer4(model):
    # V1 only unfreezes the top residual block to limit overfitting risk.
    for param in model.backbone.layer4.parameters():
        param.requires_grad = True


model = ModelBResNet50(HEAD_SPECS, shared_dim=SHARED_DIM, dropout=DROPOUT, freeze_backbone=True).to(device)
model


In [ ]:
def compute_class_weights(frame, max_class_weight=4.0):
    weight_tensors = {}
    for head, labels in HEAD_SPECS.items():
        applicable_mask = frame["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
        values = frame.loc[applicable_mask, head].astype(str)
        counts = values.value_counts()
        total = max(len(values), 1)

        weights = []
        for label in labels:
            count = max(int(counts.get(label, 0)), 1)
            weight = total / (len(labels) * count)
            weights.append(weight)

        weights = np.array(weights, dtype=np.float32)
        weights = np.clip(weights, 1.0, max_class_weight)
        weights = weights / weights.mean()
        weight_tensors[head] = torch.tensor(weights, dtype=torch.float32)

    return weight_tensors


class_weights = compute_class_weights(train_df, max_class_weight=MAX_CLASS_WEIGHT)
for head, weights in class_weights.items():
    print(head, dict(zip(HEAD_SPECS[head], [round(float(x), 3) for x in weights])))

criteria = {
    # Each head gets its own weighted loss, and ignore_index skips non-applicable targets.
    head: nn.CrossEntropyLoss(
        weight=class_weights[head].to(device),
        ignore_index=IGNORE_INDEX,
        label_smoothing=LABEL_SMOOTHING,
    )
    for head in HEAD_SPECS
}

HEAD_LOSS_WEIGHTS = {head: 1.0 for head in HEAD_SPECS}


def build_optimizer(model, stage_name):
    if stage_name == "stage1":
        params = [
            {"params": model.projection.parameters(), "lr": STAGE1_LR},
            {"params": model.heads.parameters(), "lr": STAGE1_LR},
        ]
    elif stage_name == "stage2":
        params = [
            {"params": model.backbone.layer4.parameters(), "lr": STAGE2_BACKBONE_LR},
            {"params": model.projection.parameters(), "lr": STAGE2_HEAD_LR},
            {"params": model.heads.parameters(), "lr": STAGE2_HEAD_LR},
        ]
    else:
        raise ValueError(f"Unknown stage: {stage_name}")

    return torch.optim.AdamW(params, weight_decay=WEIGHT_DECAY)


def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
    )


In [ ]:
def compute_head_losses(logits_by_head, batch):
    losses = {}
    total_loss = torch.tensor(0.0, device=device)
    for head in HEAD_SPECS:
        targets = batch[f"{head}_target"].to(device)
        logits = logits_by_head[head]
        loss = criteria[head](logits, targets)
        losses[head] = loss
        # Total loss is the sum of the active per-head losses.
        total_loss = total_loss + (HEAD_LOSS_WEIGHTS[head] * loss)
    return total_loss, losses


def init_metric_store():
    return {
        head: {"correct": 0, "total": 0, "targets": [], "preds": []}
        for head in HEAD_SPECS
    }


def update_metrics(metric_store, logits_by_head, batch):
    for head in HEAD_SPECS:
        targets = batch[f"{head}_target"].to(device)
        mask = targets != IGNORE_INDEX
        if mask.sum().item() == 0:
            continue
        preds = logits_by_head[head].argmax(dim=1)
        # Metrics only use applicable targets, so ignored heads never pollute evaluation.
        correct = (preds[mask] == targets[mask]).sum().item()
        total = mask.sum().item()
        metric_store[head]["correct"] += correct
        metric_store[head]["total"] += total
        metric_store[head]["targets"].extend(targets[mask].detach().cpu().tolist())
        metric_store[head]["preds"].extend(preds[mask].detach().cpu().tolist())


def summarize_metrics(metric_store, majority_baselines):
    rows = []
    for head, values in metric_store.items():
        total = values["total"]
        if total == 0:
            accuracy = float("nan")
            macro_f1 = float("nan")
        else:
            accuracy = values["correct"] / total
            macro_f1 = f1_score(values["targets"], values["preds"], average="macro", zero_division=0)
        rows.append({
            "head": head,
            "accuracy": accuracy,
            "majority_baseline": majority_baselines.get(head, float("nan")),
            "macro_f1": macro_f1,
            "support": total,
        })
    return pd.DataFrame(rows)


def summarize_per_class_recall(metric_store):
    rows = []
    for head, values in metric_store.items():
        if values["total"] == 0:
            continue
        recalls = recall_score(
            values["targets"],
            values["preds"],
            labels=list(range(len(HEAD_SPECS[head]))),
            average=None,
            zero_division=0,
        )
        for idx, recall in enumerate(recalls):
            rows.append({
                "head": head,
                "label": HEAD_SPECS[head][idx],
                "recall": recall,
            })
    return pd.DataFrame(rows)


In [ ]:
def run_epoch(model, loader, majority_baselines, optimizer=None):
    train_mode = optimizer is not None
    if train_mode:
        model.train()
    else:
        model.eval()

    metric_store = init_metric_store()
    total_loss = 0.0
    num_batches = 0

    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for batch in loader:
            images = batch["image"].to(device)
            # The forward pass returns a dict of logits, one tensor per head.
            logits_by_head = model(images)
            loss, _ = compute_head_losses(logits_by_head, batch)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            num_batches += 1
            update_metrics(metric_store, logits_by_head, batch)

    avg_loss = total_loss / max(num_batches, 1)
    metrics_df = summarize_metrics(metric_store, majority_baselines)
    return avg_loss, metrics_df, metric_store


def metric_objective(metrics_df):
    return metrics_df["macro_f1"].mean()


def save_checkpoint(model, optimizer, scheduler, history, best_score, best_loss, stage_name, global_epoch):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "head_specs": HEAD_SPECS,
            "category_heads": CATEGORY_HEADS,
            "history": history,
            "best_score": best_score,
            "best_loss": best_loss,
            "stage_name": stage_name,
            "global_epoch": global_epoch,
            "config": {
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "stage1_epochs": STAGE1_EPOCHS,
                "stage2_epochs": STAGE2_EPOCHS,
                "stage1_lr": STAGE1_LR,
                "stage2_head_lr": STAGE2_HEAD_LR,
                "stage2_backbone_lr": STAGE2_BACKBONE_LR,
                "dropout": DROPOUT,
            },
        },
        CHECKPOINT_PATH,
    )


history = []
best_val_score = -1.0
best_val_loss = float("inf")
global_epoch = 0


In [ ]:
def train_stage(stage_name, epochs):
    global best_val_score, best_val_loss, global_epoch

    if stage_name == "stage1":
        freeze_backbone(model)
    elif stage_name == "stage2":
        freeze_backbone(model)
        unfreeze_backbone_layer4(model)
    else:
        raise ValueError(f"Unknown stage name: {stage_name}")

    optimizer = build_optimizer(model, stage_name)
    scheduler = build_scheduler(optimizer)
    patience_counter = 0

    for stage_epoch in range(epochs):
        global_epoch += 1
        train_loss, train_metrics, _ = run_epoch(model, train_loader, train_majority_baselines, optimizer=optimizer)
        val_loss, val_metrics, val_metric_store = run_epoch(model, val_loader, val_majority_baselines, optimizer=None)
        val_score = metric_objective(val_metrics)

        history.append({
            "global_epoch": global_epoch,
            "stage": stage_name,
            "stage_epoch": stage_epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_f1_mean": val_score,
        })

        print(f"{stage_name} | epoch {stage_epoch + 1}/{epochs} | global_epoch={global_epoch}")
        print(f"  train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1_mean={val_score:.4f}")
        print("  validation metrics")
        print(val_metrics.to_string(index=False))

        scheduler.step(val_score)

        improved = (val_score > best_val_score + 1e-4) or (abs(val_score - best_val_score) <= 1e-4 and val_loss < best_val_loss)
        if improved:
            best_val_score = val_score
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, scheduler, history, best_val_score, best_val_loss, stage_name, global_epoch)
            patience_counter = 0
            print(f"  saved checkpoint to {CHECKPOINT_PATH}")
        else:
            patience_counter += 1
            print(f"  no improvement | early stop counter = {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  early stopping triggered for {stage_name}")
                break


train_stage("stage1", STAGE1_EPOCHS)
train_stage("stage2", STAGE2_EPOCHS)


In [ ]:
history_df = pd.DataFrame(history)
history_df


## Why The Baseline Overfit

In the earlier baseline run, training loss dropped steadily while validation loss stopped improving after the first few epochs. That pattern suggests the model was fitting the training distribution faster than it was learning reusable features.

The likely causes were:
- no train-time augmentation
- a fully frozen backbone for the entire run
- class imbalance, especially on `material_family`
- checkpointing on loss alone instead of a more task-aware metric
- a weak standalone `faux` class that split the elevated-material signal too thin

This V2 notebook addresses that by adding moderate augmentation, weighted losses, optional weighted sampling, partial stage-two unfreezing, a `faux -> leather` merge, macro-F1 tracking, and early stopping.


## Why Accuracy Alone Was Misleading

Raw accuracy is not enough for this model because some heads have strong majority-class baselines. `material_family`, for example, can look superficially good by overpredicting `other`.

That is why this notebook now reports:
- majority-class baseline per head
- macro F1 per head
- per-class recall per head

For the V2 material schema, success means the merged `leather` class becomes more learnable while `other` becomes less dominant as a fallback prediction.


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_metrics, test_metric_store = run_epoch(model, test_loader, test_majority_baselines, optimizer=None)
print(f"Loaded best checkpoint from stage: {checkpoint['stage_name']} | global_epoch={checkpoint['global_epoch']}")
print(f"Test loss: {test_loss:.4f}")
test_metrics


In [ ]:
test_recall_df = summarize_per_class_recall(test_metric_store)
display(test_recall_df)

for head in HEAD_SPECS:
    y_true = test_metric_store[head]["targets"]
    y_pred = test_metric_store[head]["preds"]
    if len(y_true) == 0:
        continue
    labels = list(range(len(HEAD_SPECS[head])))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"\nConfusion matrix for {head}")
    print(pd.DataFrame(cm, index=HEAD_SPECS[head], columns=HEAD_SPECS[head]))


In [ ]:
def inspect_predictions(model, dataset, n=8):
    model.eval()
    rows = []
    with torch.no_grad():
        for idx in range(min(n, len(dataset))):
            sample = dataset[idx]
            image = sample["image"].unsqueeze(0).to(device)
            logits = model(image)
            row = {
                "image_name": sample["image_name"],
                "category_group": sample["category_group"],
            }
            for head in HEAD_SPECS:
                if not sample[f"{head}_mask"].item():
                    row[f"{head}_true"] = "ignored"
                    row[f"{head}_pred"] = "ignored"
                    continue
                true_idx = sample[f"{head}_target"].item()
                pred_idx = logits[head].argmax(dim=1).item()
                row[f"{head}_true"] = idx_to_label[head][true_idx]
                row[f"{head}_pred"] = idx_to_label[head][pred_idx]
            rows.append(row)
    return pd.DataFrame(rows)


inspect_predictions(model, val_dataset, n=8)


## Optional Next Experiments

If the merged-material V2 notebook still underperforms, the next most likely improvements are:
- tuning the weighted sampler strength instead of always sampling fully inversely
- simplifying `material_family` further if `other` still dominates too much
- trying a lighter backbone like `resnet18` for faster iteration
- moving from notebook-only experimentation into a reusable training script once the recipe stabilizes
